# DeBERTa Text-Only — Inference & Evaluation (Multi-Timepoint)

Loads a saved DeBERTa regression model and predicts holistic essay scores on  
`wo_chat_test.csv` and `withchat_test.csv` — at **every source timepoint**  
(05min, 10min, 15min, 20min, 25min, full) using the mega text lookup.

**Metrics:** Pearson r, Spearman ρ, MAE, RMSE, QWK, macro F1 (on 0.5-binned scores)  
**Output:** `predictions_wo_chat.csv`, `predictions_withchat.csv`, `predictions_combined.csv`  
— each with a `source_timepoint` column.

In [ ]:
# ============================================================
# IMPORTS
# ============================================================

import numpy as np
import pandas as pd
import torch
from pathlib import Path
from scipy import stats
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    cohen_kappa_score,
    f1_score,
    classification_report,
)
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm.auto import tqdm

In [ ]:
# ============================================================
# CONFIGURATION  —  edit paths here
# ============================================================

MODEL_DIR        = Path("model")                               # folder with saved model + tokenizer
TEXT_LOOKUP_PATH  = Path("BEA Paper/text_mega_lookup.tsv")      # 5-col mega lookup (all timepoints)
WO_CHAT_CSV      = Path("BEA Paper/splits/wo_chat_test.csv")
WITHCHAT_CSV     = Path("BEA Paper/splits/withchat_test.csv")
OUTPUT_DIR       = Path("BEA Paper/predictions")

SCORE_COL  = "holistic_score"
MAX_LENGTH = 512
BATCH_SIZE = 16

# ── TIMEPOINT SELECTION ──────────────────────────────────────
# Which timepoints to evaluate. Set to a list of specific ones
# or use ALL_TIMEPOINTS to run all six.
ALL_TIMEPOINTS = ["05min", "10min", "15min", "20min", "25min", "full"]

# Examples:
#   SELECTED_TIMEPOINTS = ALL_TIMEPOINTS            # all six
#   SELECTED_TIMEPOINTS = ["full"]                   # only full texts
#   SELECTED_TIMEPOINTS = ["10min", "20min", "full"] # a subset
SELECTED_TIMEPOINTS = ALL_TIMEPOINTS

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Evaluating timepoints: {SELECTED_TIMEPOINTS}")

In [ ]:
# ============================================================
# TEXT LOOKUP
# ============================================================

def load_text_lookup(path: Path) -> dict:
    """
    Load text lookup TSV.
    Supports 4-col (filename, task, chat_used, text)
    and 5-col  (filename, task, chat_used, source_timepoint, text).
    Returns dict keyed on (filename, source_timepoint).
    """
    lookup = {}
    with open(path, 'r', encoding='utf-8') as f:
        header = f.readline().strip()
        n_cols = len(header.split('\t'))
        for line in f:
            parts = line.rstrip('\n').split('\t', n_cols - 1)
            if n_cols == 5 and len(parts) == 5:
                filename, task, chat_used, source_timepoint, text_escaped = parts
            elif n_cols == 4 and len(parts) == 4:
                filename, task, chat_used, text_escaped = parts
                source_timepoint = 'full'
            else:
                continue
            text = (
                text_escaped.replace('\\n', '\n')
                           .replace('\\r', '\r')
                           .replace('\\t', '\t')
                           .replace('\\\\', '\\')
            )
            lookup[(filename, source_timepoint)] = text
    return lookup


text_lookup = load_text_lookup(TEXT_LOOKUP_PATH)
print(f"Loaded {len(text_lookup)} texts from {TEXT_LOOKUP_PATH}")

In [ ]:
# ============================================================
# DATASET  (multi-timepoint aware)
# ============================================================

class EssayDataset(Dataset):
    """
    Builds one sample per (essay, timepoint) combination.
    For each row in `df` and each timepoint in `timepoints`,
    looks up the corresponding text from the mega lookup.
    """
    def __init__(self, df, text_lookup, tokenizer, max_length, timepoints):
        self.tokenizer  = tokenizer
        self.max_length = max_length
        self.texts      = []
        self.scores     = []
        self.filenames  = []
        self.timepoints = []

        missing = 0
        for _, row in df.iterrows():
            fn = row['filename'].strip()
            score = float(row[SCORE_COL])

            for tp in timepoints:
                text = text_lookup.get((fn, tp))
                if text is None:
                    missing += 1
                    continue

                self.texts.append(text)
                self.scores.append(score)
                self.filenames.append(fn)
                self.timepoints.append(tp)

        if missing:
            print(f"  Warning: {missing} (filename, timepoint) pairs not found in lookup")

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels':         torch.tensor(self.scores[idx], dtype=torch.float32),
        }

In [ ]:
# ============================================================
# LOAD MODEL & TOKENIZER
# ============================================================

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))
model = AutoModelForSequenceClassification.from_pretrained(
    str(MODEL_DIR),
    num_labels=1,
    problem_type="regression",
)
model = model.to(device).eval()
print(f"Model loaded from {MODEL_DIR}")

In [ ]:
# ============================================================
# PREDICTION
# ============================================================

@torch.no_grad()
def predict(dataset, model, batch_size=BATCH_SIZE):
    """Run inference and return raw prediction array."""
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    preds = []
    for batch in tqdm(loader, desc="Predicting"):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds.append(outputs.logits.squeeze(-1).cpu().numpy())
    return np.concatenate(preds)

In [ ]:
# ============================================================
# METRICS
# ============================================================

def bin_to_half(values):
    """Round to nearest 0.5."""
    return (np.array(values) * 2).round() / 2

ALL_LABELS = [f"{x:.1f}" for x in np.arange(0, 5.5, 0.5)]


def compute_metrics(y_true, y_pred) -> dict:
    """Compute regression + classification metrics."""
    if len(y_true) < 2:
        return {'n': len(y_true)}

    pearson_r, pearson_p   = stats.pearsonr(y_true, y_pred)
    spearman_r, spearman_p = stats.spearmanr(y_true, y_pred)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    # Bin to 0.5 steps for QWK and F1
    y_true_binned = bin_to_half(y_true)
    y_pred_binned = bin_to_half(np.clip(y_pred, 0.0, 5.0))

    y_true_str = [f"{x:.1f}" for x in y_true_binned]
    y_pred_str = [f"{x:.1f}" for x in y_pred_binned]

    qwk = cohen_kappa_score(y_true_str, y_pred_str,
                            weights='quadratic', labels=ALL_LABELS)

    f1_macro    = f1_score(y_true_str, y_pred_str, labels=ALL_LABELS,
                           average='macro', zero_division=0)
    f1_weighted = f1_score(y_true_str, y_pred_str, labels=ALL_LABELS,
                           average='weighted', zero_division=0)

    return {
        'n':           len(y_true),
        'Pearson r':   pearson_r,
        'Pearson p':   pearson_p,
        'Spearman ρ':  spearman_r,
        'Spearman p':  spearman_p,
        'MAE':         mae,
        'RMSE':        rmse,
        'QWK':         qwk,
        'F1 (macro)':  f1_macro,
        'F1 (weighted)': f1_weighted,
    }


def print_metrics(metrics, title=""):
    if title:
        print(f"\n{'─'*50}")
        print(f"  {title}  (n={metrics['n']})")
        print(f"{'─'*50}")
    if 'Pearson r' not in metrics:
        print("  (too few samples)")
        return
    print(f"  Pearson r:      {metrics['Pearson r']:.4f}  (p={metrics['Pearson p']:.2e})")
    print(f"  Spearman ρ:     {metrics['Spearman ρ']:.4f}  (p={metrics['Spearman p']:.2e})")
    print(f"  MAE:            {metrics['MAE']:.4f}")
    print(f"  RMSE:           {metrics['RMSE']:.4f}")
    print(f"  QWK:            {metrics['QWK']:.4f}")
    print(f"  F1 (macro):     {metrics['F1 (macro)']:.4f}")
    print(f"  F1 (weighted):  {metrics['F1 (weighted)']:.4f}")

In [ ]:
# ============================================================
# LOAD DATA & CREATE DATASETS
# ============================================================

df_wo   = pd.read_csv(WO_CHAT_CSV).dropna(subset=[SCORE_COL])
df_with = pd.read_csv(WITHCHAT_CSV).dropna(subset=[SCORE_COL])

print(f"wo_chat:   {len(df_wo)} rows  × {len(SELECTED_TIMEPOINTS)} timepoints")
print(f"withchat:  {len(df_with)} rows × {len(SELECTED_TIMEPOINTS)} timepoints")

ds_wo   = EssayDataset(df_wo,   text_lookup, tokenizer, MAX_LENGTH, SELECTED_TIMEPOINTS)
ds_with = EssayDataset(df_with, text_lookup, tokenizer, MAX_LENGTH, SELECTED_TIMEPOINTS)

print(f"\nDatasets created — wo_chat: {len(ds_wo)}, withchat: {len(ds_with)}")

In [ ]:
# ============================================================
# RUN PREDICTIONS
# ============================================================

preds_wo   = predict(ds_wo,   model)
preds_with = predict(ds_with, model)

In [ ]:
# ============================================================
# EVALUATE — per condition × timepoint + combined
# ============================================================

def evaluate_by_timepoint(filenames, timepoints, y_true, y_pred, condition):
    """Compute and print metrics for each timepoint within a condition."""
    results = []
    arr_tp = np.array(timepoints)
    for tp in SELECTED_TIMEPOINTS:
        mask = arr_tp == tp
        if mask.sum() == 0:
            continue
        m = compute_metrics(y_true[mask], y_pred[mask])
        m['condition'] = condition
        m['source_timepoint'] = tp
        print_metrics(m, f"{condition} — {tp}")
        results.append(m)

    # Overall for this condition (across all timepoints)
    m_all = compute_metrics(y_true, y_pred)
    m_all['condition'] = condition
    m_all['source_timepoint'] = 'all'
    print_metrics(m_all, f"{condition} — ALL timepoints pooled")
    results.append(m_all)
    return results


results = []
results += evaluate_by_timepoint(
    ds_wo.filenames, ds_wo.timepoints,
    np.array(ds_wo.scores), preds_wo, 'wo_chat')

results += evaluate_by_timepoint(
    ds_with.filenames, ds_with.timepoints,
    np.array(ds_with.scores), preds_with, 'withchat')

# Grand combined
y_true_all = np.concatenate([ds_wo.scores, ds_with.scores])
y_pred_all = np.concatenate([preds_wo, preds_with])
tp_all     = ds_wo.timepoints + ds_with.timepoints

arr_tp_all = np.array(tp_all)
for tp in SELECTED_TIMEPOINTS:
    mask = arr_tp_all == tp
    if mask.sum() == 0:
        continue
    m = compute_metrics(y_true_all[mask], y_pred_all[mask])
    m['condition'] = 'combined'
    m['source_timepoint'] = tp
    print_metrics(m, f"COMBINED — {tp}")
    results.append(m)

m_grand = compute_metrics(y_true_all, y_pred_all)
m_grand['condition'] = 'combined'
m_grand['source_timepoint'] = 'all'
print_metrics(m_grand, "COMBINED — ALL timepoints pooled")
results.append(m_grand)

In [ ]:
# ============================================================
# OPTIONAL: per-class classification reports (per timepoint)
# ============================================================

for label, ds, y_p in [
    ("wo_chat",  ds_wo,   preds_wo),
    ("withchat", ds_with, preds_with),
]:
    arr_tp = np.array(ds.timepoints)
    y_t = np.array(ds.scores)

    for tp in SELECTED_TIMEPOINTS:
        mask = arr_tp == tp
        if mask.sum() == 0:
            continue
        yt_str = [f"{x:.1f}" for x in bin_to_half(y_t[mask])]
        yp_str = [f"{x:.1f}" for x in bin_to_half(np.clip(y_p[mask], 0.0, 5.0))]
        print(f"\n{'='*60}")
        print(f"Classification report — {label} — {tp}")
        print(f"{'='*60}")
        print(classification_report(yt_str, yp_str, labels=ALL_LABELS, zero_division=0))

In [ ]:
# ============================================================
# SAVE PREDICTIONS
# ============================================================

def save_predictions(ds, y_pred, condition, path):
    df = pd.DataFrame({
        'filename':         ds.filenames,
        'condition':        condition,
        'source_timepoint': ds.timepoints,
        'true_score':       ds.scores,
        'predicted_score':  y_pred,
        'predicted_binned': bin_to_half(np.clip(y_pred, 0.0, 5.0)),
    })
    df.to_csv(path, index=False)
    print(f"Saved {len(df)} predictions → {path}")
    return df


df_pred_wo = save_predictions(
    ds_wo,   preds_wo,   'wo_chat',
    OUTPUT_DIR / 'predictions_wo_chat.csv')

df_pred_with = save_predictions(
    ds_with, preds_with, 'withchat',
    OUTPUT_DIR / 'predictions_withchat.csv')

df_pred_all = pd.concat([df_pred_wo, df_pred_with], ignore_index=True)
df_pred_all.to_csv(OUTPUT_DIR / 'predictions_combined.csv', index=False)
print(f"Saved {len(df_pred_all)} predictions → {OUTPUT_DIR / 'predictions_combined.csv'}")

In [ ]:
# ============================================================
# SUMMARY TABLE
# ============================================================

summary = pd.DataFrame(results)
cols_first = ['condition', 'source_timepoint', 'n']
cols_rest  = [c for c in summary.columns if c not in cols_first]
summary = summary[cols_first + cols_rest].set_index(['condition', 'source_timepoint'])

display(summary.round(4))

summary.to_csv(OUTPUT_DIR / 'metrics_summary.csv')
print(f"\nMetrics summary → {OUTPUT_DIR / 'metrics_summary.csv'}")